In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import os
import re
import json
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import roc_auc_score, roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay, classification_report

In [ ]:
report_directory = r"../CXR/files" # Files not sent for submission due to it's large size
path_labels = r'datasets/mimic-cxr-2.0.0-chexpert.csv'

- Created config cell and imported libraries at once

In [9]:
# Config 
TARGET_LABELS = ["Pleural Effusion", "Atelectasis", "Pneumonia"] 
MAX_LEN   = 128
EMBED_DIM = 128
HIDDEN    = 256
LAYERS    = 2
DROPOUT   = 0.35
BATCH     = 32
LR        = 1e-3
EPOCHS    = 30
PATIENCE  = 5
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
r_seed = 99
print("Device:", DEVICE)

In [10]:
labels_df = pd.read_csv(path_labels)
labels_df.head()

In [11]:
# Reproducibality
torch.manual_seed(42)
np.random.seed(42)

### Preprocessing the Label file

- Replace -1 with 0 and fill NaN with 0
- Convert into `int` for all columns

In [12]:
labels_df = labels_df.fillna(0).replace(-1, 0).astype('int32')

In [13]:
labels_df['No Finding'].value_counts()

- `1`s here denotes there is no finding, so removing those rows with no findings
- And it even with those having `0`s have no meaning, so removing the column as a whole

In [14]:
labels_df = labels_df[labels_df["No Finding"] != 1]
labels_df = labels_df.drop(columns='No Finding')

- The text files contain subject_id above 11000000 upto 12000000, so filtering out those out of this range

In [15]:
labels_df = labels_df[(labels_df['subject_id'] > 11000000) & (labels_df['subject_id'] < 12000000)]
print(f"Number of columns after filtering: {labels_df.shape[0]}")

### Preprocessing the `.txt` files

In [ ]:
# Takes few mins to run
rows = []

for root, dirs, files in os.walk(report_directory):
    for file in files:

        if file.endswith(".txt"):
            txt_file_path = os.path.join(root, file)
            subject_folder = os.path.basename(root)
            subject_id = subject_folder.replace("p", "")
            study_id = file.replace("s", "").replace(".txt", "")

            # Read report
            with open(txt_file_path, "r", encoding="utf-8") as f:
                text = f.read()
            
            # Extract FINDINGS
            findings_match = re.search(r'FINDINGS:\s*(.*?)(?=IMPRESSION:|$)', text, re.DOTALL)
            findings = (findings_match.group(1).strip().replace("\n", " ") if findings_match else None)

            # Extract IMPRESSION
            impression_match = re.search(r'IMPRESSION:\s*(.*)', text,re.DOTALL)
            impression = (impression_match.group(1).strip().replace("\n", " ") if impression_match else None)

            # Store row
            rows.append({
                "subject_id": subject_id,
                "study_id": study_id,
                "Findings": findings,
                "Impression": impression
            })

cxr_df = pd.DataFrame(rows)
# cxr_df.to_csv("cxr_df.csv", index=False)

In [16]:
cxr_df = pd.read_csv('datasets/cxr_df.csv')

In [17]:
cxr_df.head(3)

In [18]:
print(cxr_df[["Findings", "Impression"]].isna().sum())

- Rather than using column adding both in a single column for ease

In [20]:
cxr_df["text"] = cxr_df["Findings"].fillna("") + " " + cxr_df["Impression"].fillna("")
cxr_df["text"] = cxr_df["text"].str.strip()

In [21]:
cxr_df = cxr_df.drop(columns=['Findings','Impression'])

In [22]:
import re

def clean_text(text):
    
    text = str(text).lower()
    text = text.replace("\n", " ")
    
    # Remove placeholders like ___ or [** **]
    text = re.sub(r'_{2,}', '', text)
    text = re.sub(r'\[\*\*.*?\*\*\]', '', text)
    text = re.sub(r'\d+', '', text) # Numbers
    text = re.sub(r'\s+', ' ', text) # Normalize whitespaces
    text = re.sub(r'[^a-z\s]', '', text)
    
    return text.strip()

In [23]:
cxr_df['subject_id'] = cxr_df['subject_id'].astype('int32')
cxr_df['study_id'] = cxr_df['study_id'].astype('int32')

In [24]:
cxr_df['text'] = cxr_df['text'].apply(clean_text)

### Merging

- Merging `cxr_df` and `labels_df` on basis of `subject_id` & `study_id`

In [25]:
merged_df = cxr_df.merge(labels_df, on=["subject_id", "study_id"], how="inner")

In [87]:
# merged_df.to_csv("datasets/merged_df.csv", index=False)

In [26]:
merged_df = pd.read_csv('datasets/merged_df.csv')

### Training

In [27]:
X = merged_df['text']

In [28]:
y = merged_df.drop(columns=['text','subject_id','study_id'])

In [29]:
y.sum().sort_values(ascending=False) 

- Patient-level split  to prevents data leakage across patients

In [30]:
from sklearn.model_selection import train_test_split

subject_ids = merged_df["subject_id"].unique()
train_ids, temp_ids = train_test_split(subject_ids, test_size=0.3, random_state=r_seed)
val_ids,   test_ids = train_test_split(temp_ids,    test_size=0.5, random_state=r_seed)

train_df = merged_df[merged_df["subject_id"].isin(train_ids)].reset_index(drop=True)
val_df   = merged_df[merged_df["subject_id"].isin(val_ids)].reset_index(drop=True)
test_df  = merged_df[merged_df["subject_id"].isin(test_ids)].reset_index(drop=True)

print(f"Train: {len(train_df)} -- Val: {len(val_df)} --  Test: {len(test_df)}")

In [31]:
word_counts = Counter()
for text in train_df["text"].dropna():
    word_counts.update(text.split())

vocab = {"<PAD>": 0, "<UNK>": 1}
for word, _ in word_counts.most_common():
    vocab[word] = len(vocab)

print(f"Vocab size: {len(vocab)}")

In [32]:
# Tokenize 
def tokenize(text, vocab, max_len):
    indices = [vocab.get(t, vocab["<UNK>"]) for t in str(text).split()][:max_len] # Convert words to vocab indices
    indices += [vocab["<PAD>"]] * (max_len - len(indices))  # Pad
    return indices

for df in [train_df, val_df, test_df]:
    df["tokens"] = df["text"].apply(lambda x: tokenize(x, vocab, max_len=MAX_LEN))

In [33]:
# Dataset & DataLoaders
class CXRDataset(Dataset):
    def __init__(self, df):
        self.tokens = torch.tensor(df["tokens"].tolist(), dtype=torch.long)
        self.labels = torch.tensor(df[TARGET_LABELS].values, dtype=torch.float32)

    def __len__(self):  return len(self.tokens)
    def __getitem__(self, i):  return self.tokens[i], self.labels[i]

train_loader = DataLoader(CXRDataset(train_df), batch_size=32, shuffle=True)
val_loader   = DataLoader(CXRDataset(val_df),   batch_size=32, shuffle=False)
test_loader  = DataLoader(CXRDataset(test_df),  batch_size=32, shuffle=False)

In [34]:
# Class weights for imbalanced labels
y_train = train_df[TARGET_LABELS].values
pos_weight = torch.tensor((y_train == 0).sum(0) / (y_train == 1).sum(0),dtype=torch.float32).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print("pos_weight:", pos_weight)

In [35]:
# BiLSTM model
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden, layers, num_labels, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden,
            num_layers=layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if layers > 1 else 0)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden * 2, num_labels)

    def forward(self, x):
        x, (h, _) = self.lstm(self.embedding(x))
        # Concatenate final forward and backward hidden states
        h = torch.cat((h[-2], h[-1]), dim=1)
        return self.fc(self.drop(h))

model = BiLSTM(len(vocab), EMBED_DIM, HIDDEN, LAYERS, len(TARGET_LABELS), DROPOUT).to(DEVICE)
optimizer = Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
print(model)

In [36]:
# Training loop
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, preds, labels = 0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for tok, lab in loader:
            tok, lab = tok.to(DEVICE), lab.to(DEVICE)
            logits = model(tok)
            loss   = criterion(logits, lab)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item()
            preds.append(torch.sigmoid(logits).detach().cpu().numpy())
            labels.append(lab.cpu().numpy())
    p, l = np.vstack(preds), np.vstack(labels)
    return total_loss / len(loader), roc_auc_score(l, p, average="macro")

- The following cell ran in Kaggle and the parameters and metrics downloaded and added locally. Hence, the output not generated.
- The link for Kaggle: https://www.kaggle.com/code/saugat1909/cxr-xg/notebook

In [ ]:
train_losses, val_losses = [], []
train_aucs,   val_aucs   = [], []
best_val_loss, patience_counter = float("inf"), 0

for epoch in range(EPOCHS):
    tr_loss, tr_auc = run_epoch(train_loader, train=True)
    vl_loss, vl_auc = run_epoch(val_loader,   train=False)
    scheduler.step(vl_loss)

    train_losses.append(tr_loss);  val_losses.append(vl_loss)
    train_aucs.append(tr_auc);     val_aucs.append(vl_auc)

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | "
          f"Train  Loss {tr_loss:.4f}  AUC {tr_auc:.4f} | "
          f"Val  Loss {vl_loss:.4f}  AUC {vl_auc:.4f}")

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        patience_counter = 0
        torch.save(model.state_dict(), "BiLSTM_best.pth")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping")
            break

In [37]:
# Naive Bayes baseline (TF-IDF)
tfidf = TfidfVectorizer(max_features=6660)
X_train = tfidf.fit_transform(train_df["text"].fillna(""))
X_test  = tfidf.transform(test_df["text"].fillna(""))

nb_models, nb_aucs_dict = {}, {}
for label in TARGET_LABELS:
    nb = MultinomialNB().fit(X_train, train_df[label].values)
    nb_models[label] = nb
    proba = nb.predict_proba(X_test)[:, 1]
    nb_aucs_dict[label] = roc_auc_score(test_df[label].values, proba)
    print(f"{label:20s}  AUC: {nb_aucs_dict[label]:.4f}")

In [38]:
# Logistic Regression baseline (TF-IDF) 
from sklearn.linear_model import LogisticRegression

lr_models, lr_aucs_dict = {}, {}

for label in TARGET_LABELS:
    lr = LogisticRegression(max_iter=1000, class_weight="balanced", C=1.0)
    lr.fit(X_train, train_df[label].values)
    lr_models[label] = lr

    proba = lr.predict_proba(X_test)[:, 1]
    lr_aucs_dict[label] = roc_auc_score(test_df[label].values, proba)
    print(f"{label:<20}  AUC: {lr_aucs_dict[label]:.4f}")

- Trained in Kaggle, Loaded locally

In [39]:
# BiLSTM test predictions
model.load_state_dict(torch.load(r"models/BiLSTM_best.pth",weights_only=True,map_location=torch.device('cpu')))
model.eval()
preds, labels = [], []
with torch.no_grad():
    for tok, lab in test_loader:
        preds.append(torch.sigmoid(model(tok.to(DEVICE))).cpu().numpy())
        labels.append(lab.numpy())
all_preds  = np.vstack(preds)
all_labels = np.vstack(labels)

In [40]:
with open("models/metrics.json", "r") as f:
    history = json.load(f)

train_losses = history["train_losses"]
val_losses   = history["val_losses"]
train_aucs   = history["train_aucs"]
val_aucs     = history["val_aucs"]

### Plots

In [41]:
# Plot training and validation metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(train_losses, label="Train")
ax1.plot(val_losses, label="Val")
ax1.set_title("Loss")
ax1.legend()

ax2.plot(train_aucs, label="Train")
ax2.plot(val_aucs, label="Val")
ax2.set_title("AUC")
ax2.legend()

plt.show()

In [42]:
# ROC curves
fig, axes = plt.subplots(1, len(TARGET_LABELS), figsize=(5 * len(TARGET_LABELS), 4))

for i, label in enumerate(TARGET_LABELS):
    y_true = all_labels[:, i]

    # BiLSTM
    fpr, tpr, _ = roc_curve(y_true, all_preds[:, i])
    axes[i].plot(fpr, tpr, label="BiLSTM")

    # Naive Bayes
    nb_probs = nb_models[label].predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_true, nb_probs)
    axes[i].plot(fpr, tpr, label="NB")

    # Logistic Regression
    lr_probs = lr_models[label].predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_true, lr_probs)
    axes[i].plot(fpr, tpr, label="LR")

    # Random baseline
    axes[i].plot([0, 1], [0, 1], "k--")
    axes[i].set_title(label)
    axes[i].legend()

plt.tight_layout()
plt.show()

In [43]:
# Confusion Matrices with BiLSTM threshold = 0.5
fig, axes = plt.subplots(1, len(TARGET_LABELS), figsize=(5 * len(TARGET_LABELS), 4))
for i, label in enumerate(TARGET_LABELS):
    y_pred = (all_preds[:, i] >= 0.5).astype(int)
    cm = confusion_matrix(all_labels[:, i], y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["Neg", "Pos"]).plot(ax=axes[i], colorbar=False)
    axes[i].set_title(label)
plt.suptitle("Confusion Matrices for BiLSTM with threshold of 0.5")
plt.tight_layout()
plt.show()

In [44]:
print(classification_report(all_labels, (all_preds >= 0.5).astype(int),target_names=TARGET_LABELS, zero_division=0))

### AI usage
- Used for debugging codes and learning about the architectures themselves.
- Also to clean the code.

### Reference
1. https://docs.pytorch.org/docs/2.12/generated/torch.nn.LSTM.html
2. https://youtu.be/k2NSm3MNdYg : Deep Learning Playlist